# Module 7.2 — Memory Management

### Python Mastery · Track 7: Performance, Internals & C Extensions

Understanding how Python manages memory helps you write programs that use less of it and avoid subtle leaks. This module covers reference counting, the cyclic garbage collector, and practical techniques to reduce memory: `__slots__`, `weakref`, generators, and `array.array`.

**How to use this notebook**

- Read each explanation, then run the code cell beneath it.
- Everything here uses the standard library (`sys`, `gc`, `weakref`, `array`) and runs directly in cells.
- Attempt the exercises before consulting the solutions.

### Learning objectives

After completing this module you will be able to:

1. Explain reference counting and inspect reference counts.
2. Describe the cyclic garbage collector and control it with `gc`.
3. Reduce per-instance memory with `__slots__`.
4. Avoid reference cycles and caches that leak with `weakref`.
5. Handle large data with generators and `array.array`.

**Prerequisites:** Tracks 1 to 6, and Module 7.1.

---

## Part 1 · Reference Counting

CPython's primary memory strategy is **reference counting**: every object tracks how many references point to it, and when that count reaches zero the object is freed immediately. Assigning, passing, or storing an object increases its count; deleting a name or letting it go out of scope decreases it. `sys.getrefcount` reports the count, though it reads one higher than expected because the argument itself is a temporary reference.

In [ ]:
import sys

data = [1, 2, 3]
# getrefcount shows one extra because passing 'data' as an argument is itself a reference.
print("after creation:", sys.getrefcount(data))

another = data                 # a second name for the same list
print("after second name:", sys.getrefcount(data))

container = [data, data]       # two more references, inside the list
print("after storing twice:", sys.getrefcount(data))

del another                    # remove one reference
print("after deleting one:", sys.getrefcount(data))

When the last reference goes away, CPython reclaims the object at once. We can observe this with a `__del__` method, which runs at the moment an object is destroyed. Reference counting is prompt and predictable, but it cannot handle reference cycles, which is where the garbage collector comes in.

In [ ]:
class Tracked:
    def __init__(self, name):
        self.name = name
    def __del__(self):
        print(f"{self.name} is being freed")

obj = Tracked("A")
print("created A")
ref = obj                      # two references now
del obj                        # one reference remains; not freed yet
print("deleted first name; A still alive")
del ref                        # last reference gone; freed immediately
print("deleted second name")

## Part 2 · The Cyclic Garbage Collector

Reference counting alone cannot free a **cycle**: if object A refers to B and B refers back to A, their counts never reach zero even when nothing else points to them. CPython adds a **cyclic garbage collector** (the `gc` module) that periodically finds and frees such unreachable cycles. You can trigger it manually with `gc.collect()`, which returns how many objects it reclaimed.

In [ ]:
import gc

class Node:
    def __init__(self, name):
        self.name = name
        self.partner = None
    def __del__(self):
        print(f"freeing {self.name}")

# Create a reference cycle: each node points at the other.
a = Node("A")
b = Node("B")
a.partner = b
b.partner = a

# Dropping our names leaves the cycle unreachable, but reference counting
# cannot free it, because each still has a reference from the other.
del a, b
print("names deleted; the cycle is unreachable but not yet freed")

# The cyclic collector finds and frees it.
collected = gc.collect()
print(f"gc.collect() reclaimed objects (count >= 2): {collected >= 2}")

The `gc` module lets you inspect and tune collection: `gc.get_count()` shows how full the generational thresholds are, `gc.disable()`/`gc.enable()` turn automatic collection off and on (occasionally done in tight, allocation-free sections for speed), and `gc.get_objects()` lists tracked objects. Most programs never need to touch these, but they are valuable for diagnosing leaks.

In [ ]:
import gc

print("gc is enabled:", gc.isenabled())
print("collection counts (gen0, gen1, gen2):", gc.get_count())
print("generational thresholds:", gc.get_threshold())

# Objects that form cycles are tracked; simple objects without references are not.
print("number of objects gc is tracking:", len(gc.get_objects()) > 0)

## Part 3 · Reducing Memory with `__slots__`

By default every instance stores its attributes in a per-instance dictionary (`__dict__`), which is flexible but uses noticeable memory. Defining `__slots__` tells Python the fixed set of attributes, so it stores them compactly without a dictionary. For programs that create many small objects, this can cut memory substantially. The trade-off is that you can no longer add attributes not listed in `__slots__`.

In [ ]:
import sys

class PointDict:
    """A normal class: attributes live in a per-instance __dict__."""
    def __init__(self, x, y):
        self.x = x
        self.y = y

class PointSlots:
    """With __slots__: attributes are stored compactly, no per-instance dict."""
    __slots__ = ("x", "y")
    def __init__(self, x, y):
        self.x = x
        self.y = y

d = PointDict(1, 2)
s = PointSlots(1, 2)

# The dict-based instance carries an extra dictionary; the slots-based one does not.
print("PointDict has __dict__:", hasattr(d, "__dict__"))
print("PointSlots has __dict__:", hasattr(s, "__dict__"))
print("size of the instance dict:", sys.getsizeof(d.__dict__), "bytes")
print("slots instance has no such dict, saving that memory per object")

# The limitation: you cannot set an unlisted attribute on a slots instance.
try:
    s.z = 3
except AttributeError as e:
    print("slots restriction:", e)

## Part 4 · `weakref`: References That Do Not Keep Objects Alive

A normal reference keeps an object alive. Sometimes you want to refer to an object **without** preventing its collection, for example in a cache that should not stop unused objects from being freed. A **weak reference** (`weakref`) does exactly this: it points to an object but does not increase its reference count, so the object can still be collected, after which the weak reference reports `None`.

In [ ]:
import weakref

class Resource:
    def __init__(self, name):
        self.name = name

obj = Resource("database connection")
weak = weakref.ref(obj)              # a weak reference: does not keep obj alive

print("while the strong reference exists:", weak().name)

del obj                              # drop the only strong reference
# The object can now be collected; the weak reference yields None.
print("after deleting the strong reference:", weak())

A common, practical use is `weakref.WeakValueDictionary`, a cache whose entries vanish automatically once nothing else references the value. This prevents the classic leak where a cache holds objects alive forever.

In [ ]:
import weakref

class Image:
    def __init__(self, name):
        self.name = name

# A cache that does not keep its values alive on its own.
cache = weakref.WeakValueDictionary()

img = Image("photo.png")
cache["photo"] = img
print("in cache while referenced:", "photo" in cache)

del img                              # remove the only strong reference
# The cache entry disappears automatically, since nothing else holds the Image.
print("in cache after the object is gone:", "photo" in cache)

## Part 5 · Large Data: Generators and `array.array`

Two techniques keep memory low when handling large quantities of data. A **generator** produces items one at a time instead of building a whole list, so memory stays flat regardless of the count. And `array.array` stores numbers in a compact C-level buffer rather than as full Python objects, using far less memory than a list for large numeric sequences.

In [ ]:
import sys

# A list holds every value at once; a generator holds only the current one.
def squares_list(n):
    return [i * i for i in range(n)]        # all n values in memory

def squares_gen(n):
    return (i * i for i in range(n))         # produces one value at a time

big_list = squares_list(100_000)
gen = squares_gen(100_000)

print("list size:     ", sys.getsizeof(big_list), "bytes (holds all values)")
print("generator size:", sys.getsizeof(gen), "bytes (holds almost nothing)")
print("both yield the same first three:", big_list[:3], "vs", [next(gen) for _ in range(3)])

In [ ]:
import sys
from array import array

# A list of ints stores full Python objects; array packs raw machine integers.
int_list = list(range(100_000))
int_array = array("i", range(100_000))       # 'i' = signed C int

print("list of ints: ", sys.getsizeof(int_list), "bytes")
print("array of ints:", sys.getsizeof(int_array), "bytes")
print("the array is far smaller because it avoids per-element object overhead")

# An array behaves like a sequence for most purposes.
print("sum via array:", sum(int_array[:10]), "| first five:", int_array[:5].tolist())

---

## Worked Examples

| Examples | Level |
|---|---|
| 1 and 2 | Easy |
| 3 and 4 | Medium |
| 5 and 6 | Difficult |

### Example 1 — Watching reference counts change (Easy)

In [ ]:
import sys
x = [10, 20]
print("start:", sys.getrefcount(x))
y = x
print("after alias:", sys.getrefcount(x))
del y
print("after del:", sys.getrefcount(x))

### Example 2 — A slots class refuses unknown attributes (Easy)

In [ ]:
class Compact:
    __slots__ = ("a", "b")
    def __init__(self, a, b):
        self.a, self.b = a, b

c = Compact(1, 2)
print("a, b:", c.a, c.b)
try:
    c.c = 3
except AttributeError as e:
    print("rejected:", e)

### Example 3 — Breaking a cycle with the collector (Medium)

In [ ]:
import gc

class Item:
    def __init__(self): self.ref = None

gc.collect()                       # clear the slate
a, b = Item(), Item()
a.ref = b
b.ref = a                          # cycle
del a, b
freed = gc.collect()               # collector frees the unreachable cycle
print("objects reclaimed by gc:", freed)

### Example 4 — A weak-value cache that self-cleans (Medium)

In [ ]:
import weakref

class Record:
    def __init__(self, key): self.key = key

cache = weakref.WeakValueDictionary()
r = Record("alpha")
cache["alpha"] = r
print("present:", list(cache.keys()))
del r
print("after release:", list(cache.keys()))

### Example 5 — Memory comparison: list versus generator pipeline (Difficult)

In [ ]:
import sys

# Goal: sum the squares of even numbers up to a large n, using little memory.
n = 1_000_000

# List-based: builds large intermediate lists.
def with_lists(n):
    evens = [i for i in range(n) if i % 2 == 0]
    squares = [e * e for e in evens]
    return sum(squares)

# Generator-based: streams values, building nothing large.
def with_generators(n):
    evens = (i for i in range(n) if i % 2 == 0)
    squares = (e * e for e in evens)
    return sum(squares)

assert with_lists(1000) == with_generators(1000)     # same answer on a small case
print("generator pipeline result:", with_generators(n))
print("the generator version never materialises the full lists, so memory stays flat")

### Example 6 — Compact numeric storage with array (Difficult)

In [ ]:
import sys
from array import array

# Store one million temperatures as floats: list versus array.
readings_list = [20.0 + (i % 100) * 0.1 for i in range(1_000_000)]
readings_array = array("d", readings_list)     # 'd' = double-precision float

print("list size: ", sys.getsizeof(readings_list), "bytes")
print("array size:", sys.getsizeof(readings_array), "bytes")
print(f"the array uses about {sys.getsizeof(readings_list)/sys.getsizeof(readings_array):.1f}x less")

# Statistics still work directly on the array.
print("average:", round(sum(readings_array) / len(readings_array), 2))

---

## Exercises

**Exercise 1 (Easy).** Create a list, alias it to a second name, and use `sys.getrefcount` to show the count before and after creating the alias.

In [ ]:
# Your solution here


**Exercise 2 (Easy).** Define a class with `__slots__ = ("name", "age")`, create an instance, and demonstrate that setting an unlisted attribute raises `AttributeError`.

In [ ]:
# Your solution here


**Exercise 3 (Medium).** Create two objects that reference each other in a cycle, delete your names for them, call `gc.collect()`, and print how many objects were reclaimed.

In [ ]:
# Your solution here


**Exercise 4 (Medium).** Use a `weakref.WeakValueDictionary` to cache an object, then show that the entry disappears after the strong reference is deleted.

In [ ]:
# Your solution here


**Exercise 5 (Difficult).** Build a list of 500,000 integers and an `array("i", ...)` of the same values, then print both sizes from `sys.getsizeof` and the ratio between them.

In [ ]:
# Your solution here


---

## Solutions

**Solution 1**

In [ ]:
import sys
items = [1, 2, 3]
print("before:", sys.getrefcount(items))
alias = items
print("after:", sys.getrefcount(items))

**Solution 2**

In [ ]:
class Person:
    __slots__ = ("name", "age")
    def __init__(self, name, age):
        self.name, self.age = name, age

p = Person("Asha", 30)
print(p.name, p.age)
try:
    p.email = "a@b.com"
except AttributeError as e:
    print("rejected:", e)

**Solution 3**

In [ ]:
import gc

class N:
    def __init__(self): self.other = None

gc.collect()
x, y = N(), N()
x.other = y; y.other = x
del x, y
print("reclaimed:", gc.collect())

**Solution 4**

In [ ]:
import weakref

class Thing:
    def __init__(self, k): self.k = k

cache = weakref.WeakValueDictionary()
t = Thing("one")
cache["one"] = t
print("before:", "one" in cache)
del t
print("after:", "one" in cache)

**Solution 5**

In [ ]:
import sys
from array import array
n = 500_000
as_list = list(range(n))
as_array = array("i", range(n))
print("list: ", sys.getsizeof(as_list), "bytes")
print("array:", sys.getsizeof(as_array), "bytes")
print(f"ratio: about {sys.getsizeof(as_list)/sys.getsizeof(as_array):.1f}x")

---

## Summary and Key Points

- CPython uses reference counting: objects are freed the instant their count hits zero. `sys.getrefcount` reports the count (one higher, due to the call itself); `__del__` runs at destruction.
- Reference counting cannot free reference cycles, so a cyclic garbage collector (`gc`) periodically reclaims them; `gc.collect()` runs it manually and `gc` exposes inspection and tuning.
- `__slots__` declares a fixed attribute set, removing the per-instance `__dict__` to save memory for many small objects, at the cost of not allowing unlisted attributes.
- `weakref` refers to an object without keeping it alive; `WeakValueDictionary` makes self-cleaning caches that do not leak by holding values forever.
- For large data, generators stream values with flat memory instead of building lists, and `array.array` stores numbers in a compact C buffer, using far less memory than a list.

### Next module: 7.3 — Python Internals

The next module explores the object model, the attribute lookup chain, and the Global Interpreter Lock, including the experimental free-threaded (no-GIL) build that points to Python's future.